In [24]:
import pandas as pd
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

In [25]:
df = pd.read_csv("mabel.txt",header=None,on_bad_lines='skip',encoding='utf8')

In [26]:
df =df.drop(0)
df.columns = ["Date","Chat"]
Message = df["Chat"].str.split("-",n=1,expand = True)
df["Time"]= Message[0]
Message1 = Message[1].str.split(":",n=1,expand = True)
df["Name"]= Message1[0]
df["Chat"]= Message1[1]
df = df[["Date","Time","Name","Chat"]]

In [27]:
df

,Date,Time,Name,Chat
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K
4,05/12/19,1:42 pm,Mabel Infoziant,Ok
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?
7,05/12/19,1:43 pm,AR❤,Yes Mam
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working
9,05/12/19,1:43 pm,AR❤,Yeah mam
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too


In [28]:
#Sentiment Analysis
dataset = df
dataset.dropna(inplace= True)

sid = SentimentIntensityAnalyzer()
senti = sid.polarity_scores(dataset['Chat'][4])
senti

{'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound': 0.296}

In [29]:
def sentimentAnalysis(data, columnname):
    import matplotlib.pyplot as plt
    import seaborn as sns
    from nltk.sentiment.vader import SentimentIntensityAnalyzer
    
    sid = SentimentIntensityAnalyzer()

    data.dropna(inplace = True)

    data['score']    = data[columnname].apply(lambda commentText: sid.polarity_scores(commentText))
    data['compound'] = data['score'].apply(lambda score_dict: score_dict['compound'])
    data['negative'] = data['score'].apply(lambda score_dict: score_dict['neg'])
    data['positive'] = data['score'].apply(lambda score_dict: score_dict['pos'])
    data['neutral']  = data['score'].apply(lambda score_dict: score_dict['neu'])

    data['comp_score'] = data['compound'].apply(lambda c: 'pos' if c>= 0 else 'neg')
    plt.clf()

    posneg = pd.DataFrame(data['comp_score'].value_counts())
    return posneg,data

In [30]:
pos, data_senti = sentimentAnalysis(dataset, columnname = 'Chat')

<Figure size 640x480 with 0 Axes>

In [31]:
data_senti

,Date,Time,Name,Chat,score,compound,negative,positive,neutral,comp_score
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos
4,05/12/19,1:42 pm,Mabel Infoziant,Ok,"{'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...",0.2960,0.000,1.000,0.000,pos
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos
7,05/12/19,1:43 pm,AR❤,Yes Mam,"{'neg': 0.0, 'neu': 0.27, 'pos': 0.73, 'compou...",0.4019,0.000,0.730,0.270,pos
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos
9,05/12/19,1:43 pm,AR❤,Yeah mam,"{'neg': 0.0, 'neu': 0.312, 'pos': 0.688, 'comp...",0.2960,0.000,0.688,0.312,pos
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos


In [32]:
#Topic modelling
tfidf = TfidfVectorizer(stop_words='english')
dtm = tfidf.fit_transform(df["Chat"])
dtm

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 188 stored elements and shape (50, 122)>

In [33]:
nmf_model = NMF(n_components = 5,random_state = 42)
nmf_model.fit(dtm)

,n_components,5
,init,None
,solver,'cd'
,beta_loss,'frobenius'
,tol,0.0001
,max_iter,200
,random_state,42
,alpha_W,0.0
,alpha_H,'same'
,l1_ratio,0.0
,verbose,0


In [34]:
for index,topic in enumerate (nmf_model.components_):
    results = ([tfidf.get_feature_names_out()[i] for i in topic.argsort()[-10:]])
    print(results)

['updates', 'okay', 'free', 'cal', 'kk', 'read', 'abstract', 'sure', 'yeah', 'mam']
['hi', 'financial', 'finalization', 'send', 'just', 'know', 'tomorrow', 'mam', 'thank', 'ok']
['start', 'details', 'abstracts', 'students', 'phone', 'ask', 'office', 'vignesh', 'number', 'send']
['plan', 'night', 'late', 'spoke', 'mabel', 'just', 'couldn', 'tomorrow', 'church', 'hi']
['talk', '30', 'today', 'rani', '7pm', 'ya', 'details', 'ramisha', 'need', 'yes']


In [35]:
topic_results = nmf_model.transform(dtm)
df["Topic"]   = topic_results.argmax(axis = 1)

In [36]:
df

,Date,Time,Name,Chat,score,compound,negative,positive,neutral,comp_score,Topic
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,3
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,0
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4
4,05/12/19,1:42 pm,Mabel Infoziant,Ok,"{'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...",0.2960,0.000,1.000,0.000,pos,1
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4
7,05/12/19,1:43 pm,AR❤,Yes Mam,"{'neg': 0.0, 'neu': 0.27, 'pos': 0.73, 'compou...",0.4019,0.000,0.730,0.270,pos,4
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,2
9,05/12/19,1:43 pm,AR❤,Yeah mam,"{'neg': 0.0, 'neu': 0.312, 'pos': 0.688, 'comp...",0.2960,0.000,0.688,0.312,pos,0
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4


In [37]:
#Flirt prediction

flirt_words=['babe', 'baby', 'honey', 'cutie', 'crush', 'sexy', 'beautiful', 'handsome', 'gorgeous', 'cute', 'hot', 
             'hang out', 'meet up', 'date', 'alone', 'just us', 'tonight', 'cal u', 'are u free', 'location']
flirt_set = set(flirt_words)

def check_for_flirtword(data,columnname):
    df['flirt_message'] = df[columnname].apply(lambda x: 'Yes' if set(str(x).lower().split()) & flirt_set else 'No')                
    flirt = pd.DataFrame(data['flirt_message'])   
    return data, flirt   

In [38]:
data,flirt = check_for_flirtword (dataset, 'Chat')
data

,Date,Time,Name,Chat,score,compound,negative,positive,neutral,comp_score,Topic,flirt_message
1,05/12/19,1:42 pm,Mabel Infoziant,Hi this is Mabel we just spoke,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,3,No
2,05/12/19,1:42 pm,Mabel Infoziant,What’s your full name,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,0,No
3,05/12/19,1:42 pm,AR❤,Ramisha Rani K,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4,No
4,05/12/19,1:42 pm,Mabel Infoziant,Ok,"{'neg': 0.0, 'neu': 0.0, 'pos': 1.0, 'compound...",0.2960,0.000,1.000,0.000,pos,1,No
5,05/12/19,1:42 pm,Mabel Infoziant,ramisharanik@gmail.com,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4,No
6,05/12/19,1:43 pm,Mabel Infoziant,Your email Id?,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4,No
7,05/12/19,1:43 pm,AR❤,Yes Mam,"{'neg': 0.0, 'neu': 0.27, 'pos': 0.73, 'compou...",0.4019,0.000,0.730,0.270,pos,4,No
8,05/12/19,1:43 pm,Mabel Infoziant,I will send 2 abstracts for u to start working,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,2,No
9,05/12/19,1:43 pm,AR❤,Yeah mam,"{'neg': 0.0, 'neu': 0.312, 'pos': 0.688, 'comp...",0.2960,0.000,0.688,0.312,pos,0,No
10,05/12/19,1:43 pm,Mabel Infoziant,Give me the list that u have too,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,0.000,1.000,pos,4,No


Who is talkative ?

In [40]:
talkative = (dataset['Name'].value_counts().idxmax()).upper()
print("More Talktative:", talkative)

less_Talktative = (dataset['Name'].value_counts().idxmin()).upper()
print("Less Talktative:", less_Talktative)

More Talktative:  AR❤
Less Talktative:  MABEL INFOZIANT


Most active date & time

In [42]:
most_active_date = dataset["Date"].value_counts().idxmax()
print("Most active date:", most_active_date)

most_active_time = dataset["Time"].value_counts().idxmax()
print("Most Active Time:", most_active_time)

Most active date: 05/12/19
Most Active Time:  1:43 pm 


Message count sent by each user

In [44]:
media =dataset[dataset["Chat"] == " <media omitted>"]

if len(media):
    media_count = pd.DataFrame(media["Name"].value_counts())
    media_share = media["Name"].value_counts().idxmax()
else:
    media_count = "0"
    media_share = "No Media were shared"
    
print("media_count shared:", media_count)
print("Who shared maximum media:", media_share)

media_count shared: 0
Who shared maximum media: No Media were shared


Missed calls

In [46]:
missed_call = dataset[dataset["Chat"] == "missed voice call"]

if len(missed_call):
    missedcall_count = pd.DataFrame(missed_call["Name"].value_counts())
    Maximum_missed_call = missed_call["Name"].value_counts().idxmax()
else:
    missedcall_count = "0"
    maximum_missed_call = "No missed voice call"
    
print("Total missed calls:", missedcall_count)
print("Most missed calls from:", maximum_missed_call)

Total missed calls: 0
Most missed calls from: No missed voice call


In [ ]:
Missed video calls

In [47]:
missed_video_call =dataset[dataset["Chat"] == "missed video call"]

if len(missed_video_call):
    missedcall_count = pd.DataFrame(missed_video_call["Name"].value_counts())
    Maximum_missed_call = missed_video_call["Name"].value_counts().idxmax()
else:
    missedcall_count = "0"
    maximum_missed_call = "No missed video call"
    
print("Total missed video call:",missedcall_count)
print("Most video missed calls from:", maximum_missed_call)

Total missed video call: 0
Most video missed calls from: No missed video call
